In [1]:
emails = [
        {
            "subject": "周末见个面？",
            "content": """
                嗨 虎哥，
                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？

                祝好，简
            """,
            "from": "jane@itcast.cn",
            "status": "unread"
        },
        {
            "subject": "周五会议",
            "content": """
                嗨 虎哥，
                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？

                祝好，小李
            """,
            "from": "lixiaolong@itcast.cn",
            "status": "checked"
        }
    ]

In [2]:
from langchain.tools import tool

# 定义工具，收取和发送邮件
@tool
def check_inbox() -> str:
    """查看收件箱中的邮件."""

    # 模拟收件箱邮件
    return emails


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """发送回复邮件"""
    return f"邮件已发送至 {to} , 主题： {subject} , 内容： {body}"

In [3]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

# 创建人工审核中间件
human_in_loop_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": {
            "description": "请确认邮件内容",
            "allowed_decisions": ["approve", "reject", "edit"]
        }
    }
)

In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

from dotenv import load_dotenv

load_dotenv()

agent_with_hitl = create_agent(
    model="deepseek-flash",
    tools=[check_inbox, send_email],
    middleware=[human_in_loop_middleware],
    checkpointer=InMemorySaver(),
    system_prompt="你是一个邮件助手"
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我查看邮件")]},
    config=config
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看收件箱中的邮件。
Tool Calls:
  check_inbox (call_00_ET_G4Kp05u8kiw4COphErxb9029)
 Call ID: call_00_ET_G4Kp05u8kiw4COphErxb9029
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱里有 **2 封邮件**：

---

**1. 周末见个面？** · 来自 jane@itcast.cn · 🔵 未读
> 嗨 虎哥，
> 我下周会去城里，不知道我们有没有机会一起喝杯咖啡？
> 
> 祝好，简

---

**2. 周五会

In [6]:
response = agent_with_hitl.invoke(
    {"messages": [HumanMessage("帮我回复Jane，很高兴她能来，问问她具体见面时间")]},
    config=config # 注意使用同一个config，也就是thread_id
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看收件箱中的邮件。
Tool Calls:
  check_inbox (call_00_ET_G4Kp05u8kiw4COphErxb9029)
 Call ID: call_00_ET_G4Kp05u8kiw4COphErxb9029
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱里有 **2 封邮件**：

---

**1. 周末见个面？** · 来自 jane@itcast.cn · 🔵 未读
> 嗨 虎哥，
> 我下周会去城里，不知道我们有没有机会一起喝杯咖啡？
> 
> 祝好，简

---

**2. 周五会

In [7]:
print(response)

{'messages': [HumanMessage(content='帮我查看邮件', additional_kwargs={}, response_metadata={}, id='668a4f0c-98ee-4efa-b8dd-06e2923addfa'), AIMessage(content='我来帮您查看收件箱中的邮件。', additional_kwargs={'refusal': None, 'reasoning_content': ''}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 350, 'total_tokens': 382, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 350}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'ce10107e-a8b2-4882-9f05-67435bef5578', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1e4d8-141b-7100-9bff-79a05bdc6164-0', tool_calls=[{'name': 'ch

In [8]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",            # 人工确认的决定，是reject
                    "message": "语气太书面了，Jane是我的女朋友！" # 请求被拒绝的原因
                }
            ]
        }
    ),
    config=config  # 相同的thread_id以恢复暂停的对话
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看收件箱中的邮件。
Tool Calls:
  check_inbox (call_00_ET_G4Kp05u8kiw4COphErxb9029)
 Call ID: call_00_ET_G4Kp05u8kiw4COphErxb9029
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱里有 **2 封邮件**：

---

**1. 周末见个面？** · 来自 jane@itcast.cn · 🔵 未读
> 嗨 虎哥，
> 我下周会去城里，不知道我们有没有机会一起喝杯咖啡？
> 
> 祝好，简

---

**2. 周五会

In [9]:
from langgraph.types import Command

response = agent_with_hitl.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config  # 相同的thread_id以恢复暂停的对话
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

帮我查看邮件
================================== Ai Message ==================================

我来帮您查看收件箱中的邮件。
Tool Calls:
  check_inbox (call_00_ET_G4Kp05u8kiw4COphErxb9029)
 Call ID: call_00_ET_G4Kp05u8kiw4COphErxb9029
  Args:
================================= Tool Message =================================
Name: check_inbox

[{"subject": "周末见个面？", "content": "\n                嗨 虎哥，\n                我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n                祝好，简\n            ", "from": "jane@itcast.cn", "status": "unread"}, {"subject": "周五会议", "content": "\n                嗨 虎哥，\n                非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n                祝好，小李\n            ", "from": "lixiaolong@itcast.cn", "status": "checked"}]
================================== Ai Message ==================================

您的收件箱里有 **2 封邮件**：

---

**1. 周末见个面？** · 来自 jane@itcast.cn · 🔵 未读
> 嗨 虎哥，
> 我下周会去城里，不知道我们有没有机会一起喝杯咖啡？
> 
> 祝好，简

---

**2. 周五会